MODULE
1. Encoder
2. Token_Representation
3. Entity FFN
4. Span FFN
5. Matching
6. Loss
7. Decder

In [1]:
BATCH_SIZE=32
SPAN_CHANNEL=100

In [2]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer,AutoModel

In [3]:
#Encoder
BackBone_Model="klue/bert-base"

In [4]:
# Input -> (BATCH_SIZE,SEQ_LEN) , Each -> (SEQ_LEN,)
# Output -> (BATCH_SIZE,SEQ_LEN,Hidden_state)
class EncoderModule(nn.Module):
    def __init__(self,backbone_model:str):
        super().__init__()
        self.encoder=AutoModel.from_pretrained(backbone_model)
        self.hidden_size=self.encoder.config.hidden_size

    def forward(self,input_ids:torch.Tensor,attention_mask:torch.Tensor)->torch.Tensor:
        outputs=self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        hidden_states=outputs.last_hidden_state

        return hidden_states

In [5]:
# Token_Representation(h generator)
# Input -> (BATCH_SIZE,SEQ_LEN,Hidden_size) , (BATCH_SIZE,SEQ_LEN)
# Output -> (BATCH_SIZE,T_MAX,Hidden_size)
class Text_Token_Represntation(nn.Module):
    def __init__(self):
        super().__init__()

    def pad_sequence(self,sentences):
        max_len=max(line.size(0) for line in sentences)
        hidden_size=sentences[0].size(1)

        pad_sequences=[]

        for sent in sentences:
            pad_len=max_len-sent.size(0)

            if pad_len>0:
                pad_tensor=torch.zeros(
                    pad_len,
                    hidden_size,
                    device=sent.device,
                    dtype=sent.dtype
                )
                sent=torch.cat([sent,pad_tensor],dim=0)

            pad_sequences.append(sent)
        return torch.stack(pad_sequences,dim=0)

    def forward(self,hidden_states,text_mask):
        text_vector=[]

        for hs,mask in zip(hidden_states,text_mask):
            h_vector=hs[mask.bool()]
            text_vector.append(h_vector)

        padded_text_vectors=self.pad_sequence(text_vector)

        return padded_text_vectors

In [6]:
# Test
hidden_states=torch.randn(2,6,4)
text_mask=torch.Tensor([
    [0,0,1,1,1,0],
    [0,1,1,0,0,0]
])
model=Text_Token_Represntation()
output=model(hidden_states,text_mask)


print("hidden_state shape : ",hidden_states.shape)
print("text_mask shape : ",text_mask.shape)
print("output shape : ",output.shape)
print(hidden_states)
#print("\n")
print(output)


hidden_state shape :  torch.Size([2, 6, 4])
text_mask shape :  torch.Size([2, 6])
output shape :  torch.Size([2, 3, 4])
tensor([[[ 0.1210, -0.6378, -0.9843, -0.2086],
         [-0.7036, -0.7394,  0.9241,  1.0185],
         [-0.4553, -0.2482,  1.2919, -1.4935],
         [ 0.2846,  0.9140,  0.8479, -0.3354],
         [-1.2026, -0.7689,  0.7688,  1.2023],
         [ 0.3041,  0.9971,  2.2871, -0.4433]],

        [[ 0.2645,  0.9534,  1.5672, -1.2241],
         [ 1.5179, -0.3890, -1.9233,  0.5665],
         [ 0.3943, -1.4510, -1.0840,  0.8395],
         [ 0.6235, -0.5443,  1.0190,  0.2544],
         [ 0.1735, -1.7460,  0.4321,  1.7381],
         [ 0.6471, -0.0351, -0.8242, -0.0407]]])
tensor([[[-0.4553, -0.2482,  1.2919, -1.4935],
         [ 0.2846,  0.9140,  0.8479, -0.3354],
         [-1.2026, -0.7689,  0.7688,  1.2023]],

        [[ 1.5179, -0.3890, -1.9233,  0.5665],
         [ 0.3943, -1.4510, -1.0840,  0.8395],
         [ 0.0000,  0.0000,  0.0000,  0.0000]]])


In [7]:
# Token_Representation(p generator)
# Input -> (BATCH_SIZE,SEQ_LEN,Hidden_size) , (BATCH_SIZE,SEQ_LEN)
# Output -> (BATCH_SIZE,M,Hidden_size) (M=[ENT] count)
class ENT_Token_Representation(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self,hidden_states,ent_mask):
        ent_vectors=[]

        for hs,mask in zip(hidden_states,ent_mask):
            p_vector=hs[mask.bool()]
            ent_vectors.append(p_vector)

        ent_vectors=torch.stack(ent_vectors,dim=0)

        return ent_vectors

In [8]:
# Test
hidden_states=torch.randn(2,8,4)
ent_token_mask=torch.tensor([
    [0,1,0,1,0,1,0,0],
    [0,1,0,1,0,1,0,0]
])
model=ENT_Token_Representation()
output=model(hidden_states,ent_token_mask)

print("hidden_states shape : ",hidden_states.shape)
print(hidden_states)
print("ent_token_mask shape : ",ent_token_mask.shape)
print("output shape : ",output.shape)
print(output)

hidden_states shape :  torch.Size([2, 8, 4])
tensor([[[-1.1791, -0.5885, -0.5659, -0.5206],
         [ 0.2274, -0.0952, -1.1195, -0.5217],
         [ 0.7129,  1.0369, -1.3976, -1.8200],
         [-0.9111, -0.1097, -0.7673,  0.1928],
         [-1.3412, -1.2766,  0.0103,  0.4448],
         [-0.7217, -1.3730, -0.2244, -0.8725],
         [ 1.3213, -0.5450, -0.3891,  0.0376],
         [ 0.3758, -1.1659, -0.7894,  0.2981]],

        [[ 1.3205,  1.9494,  0.1793, -0.4313],
         [-1.2074,  0.0561, -0.8934, -0.4914],
         [ 0.8342, -2.0757, -0.2017, -0.6437],
         [-0.2863,  0.1048,  0.0075, -0.8264],
         [ 0.5578,  1.0174, -0.2799, -0.8026],
         [-1.4091,  0.0527,  0.7232,  0.5730],
         [-0.5103,  0.6122,  1.2026,  0.7218],
         [-0.3833,  0.3093,  0.2623, -0.1687]]])
ent_token_mask shape :  torch.Size([2, 8])
output shape :  torch.Size([2, 3, 4])
tensor([[[ 0.2274, -0.0952, -1.1195, -0.5217],
         [-0.9111, -0.1097, -0.7673,  0.1928],
         [-0.7217, -1.37

In [9]:
class Linear(nn.Module):
    def __init__(self,input_x,output_y,bias=True):
        super().__init__()

        self.w=nn.Parameter(torch.randn(output_y,input_x))

        if bias:
            self.bias=nn.Parameter(torch.randn(output_y))
        else:
            self.bias=None
    def forward(self,x):
        out=x @ self.w.T

        if self.bias is not None:
            out+=self.bias
        return out

In [10]:
# test
linear=Linear(768,256)
x=torch.randn(32,10,768)
y=linear(x)
print(y.shape)

torch.Size([32, 10, 256])


In [11]:
#class ReLU(nn.Module):
#    def __init__(self):
#        super().__init__()
#    def forward(self,x):
#        if(x>=0):
#            return x
#        else:
#            return 0
# 이거는 Tensor 각각을 보지않아서 동작안함
class ReLU(nn.Module):
    def forward(self,x):
        return x*(x>0)

In [12]:
# test
activation=ReLU()
print(activation(-2))

0


In [13]:
class Sequential(nn.Module):
    def __init__(self,*layers):
        super().__init__()
        self.layers=nn.ModuleList(layers)

    def forward(self,x):
        for layer in self.layers:
            x=layer(x)

        return x

In [14]:
# test
model=Sequential(
    Linear(4,8),
    ReLU(),
    #Linear(8,2)
)
x=torch.randn(3,4)
y=model(x)
print(y.shape)
print(x)
print(y)

torch.Size([3, 8])
tensor([[ 0.0714,  0.7944, -0.9488, -0.1112],
        [-0.1433,  0.9999,  0.0909,  0.3056],
        [ 2.2365,  1.5220, -1.1578,  0.0851]])
tensor([[-0.0000, -0.0000, -0.0000, 0.0194, -0.0000, -0.0000, 1.3845, 0.3326],
        [-0.0000, -0.0000, 0.4212, 1.0672, -0.0000, -0.0000, -0.0000, -0.0000],
        [-0.0000, -0.0000, -0.0000, -0.0000, -0.0000, 2.4737, 6.4478, -0.0000]],
       grad_fn=<MulBackward0>)


In [15]:
# Entity FFN
# input -> (Batch_size,M,d_model)
# output -> (Batch_size,M,d_model)
class Entity_Representation(nn.Module):
    def __init__(self,d_model,d_ff):
        super().__init__()

        self.net=Sequential(
            Linear(d_model,d_ff),
            ReLU(),
            Linear(d_ff,d_model)
        )
    def forward(self,x):
        output=self.net(x)
        return output

In [16]:
# test
model=Entity_Representation(768,1024)
x=torch.randn(32,10,768)
y=model(x)
print(x.shape)
print(y.shape)

torch.Size([32, 10, 768])
torch.Size([32, 10, 768])


In [17]:
# Span FFN
class Span_Representation(nn.Module):
    def __init__(self,d_model,d_ff):
        super().__init__()
        self.net=Sequential(
            Linear(2*d_model,d_ff),
            ReLU(),
            Linear(d_ff,d_model)
        )

    def forward(self,start_token,end_token):
        span=torch.cat([start_token,end_token],dim=-1)
        output=self.net(span)
        return output

In [18]:
# test
model=Span_Representation(768,1024)
start_token=torch.randn(32,20,768)
end_token=torch.randn(32,20,768)
y=model(start_token,end_token)
print("start shape : ",start_token.shape)
print("end shape : ",end_token.shape)
print(y.shape)

start shape :  torch.Size([32, 20, 768])
end shape :  torch.Size([32, 20, 768])
torch.Size([32, 20, 768])


In [19]:
class Sigmoid(nn.Module):
    def forward(self,x):
        output=1/(1+torch.exp(-x))
        return output

In [41]:
class Matching(nn.Module):
    def __init__(self):
        super().__init__()
        self.sigmoid=Sigmoid()
    def forward(self,entity,span):
        entity=entity.transpose(-1,-2)
        output=span@entity
        #score=self.sigmoid(output)
        return output

In [21]:
# test
B,E,N,d=2,5,7,4
entity=torch.randn(B,E,d)
span=torch.randn(B,N,d)
model=Matching()
score=model(entity,span)
print(score.shape)
print(score)

torch.Size([2, 7, 5])
tensor([[[0.8230, 0.4777, 0.2139, 0.1785, 0.1350],
         [0.9440, 0.0351, 0.0203, 0.5841, 0.5960],
         [0.7512, 0.0356, 0.0356, 0.2003, 0.9123],
         [0.5167, 0.2707, 0.3319, 0.8087, 0.7891],
         [0.1872, 0.9144, 0.6141, 0.1328, 0.4670],
         [0.5880, 0.2677, 0.5736, 0.2525, 0.4511],
         [0.1174, 0.8672, 0.9855, 0.9813, 0.5251]],

        [[0.4443, 0.1215, 0.9674, 0.3715, 0.2454],
         [0.5488, 0.6552, 0.2776, 0.5497, 0.8926],
         [0.5247, 0.5855, 0.0022, 0.5965, 0.6282],
         [0.5060, 0.1672, 0.0293, 0.4879, 0.4540],
         [0.5068, 0.8437, 0.0266, 0.6073, 0.2842],
         [0.3936, 0.5568, 0.9338, 0.4432, 0.0060],
         [0.5247, 0.1017, 0.9179, 0.3990, 0.7577]]])


In [22]:
class BCEloss(nn.Module):
    def forward(self,pred,ans):
        eps=1e-12
        pred=torch.clamp(pred,eps,1-eps)
        loss=-(ans*torch.log(pred)+(1-ans)*torch.log(1-pred))
        output=loss.mean()
        return output

In [23]:
# test
target=torch.randint(0,2,(B,N,E)).float()
loss_f=BCEloss()
loss=loss_f(score,target)
print(loss)

tensor(1.1557)


In [24]:
class Decoder(nn.Module):
    def __init__(self,entity_types,threshold=0.5,mode="flat"):
        super().__init__()
        self.entity_types=entity_types
        self.threshold=threshold
        self.mode=mode

    def overlapping(self,span1,span2):
        s1,e1=span1
        s2,e2=span2

        return not(e1<s2 or e2<s1)

    def partial_overlap(self,span1,span2):
        s1,e1=span1
        s2,e2=span2

        overlap=not(e1<s2 or e2<s1)
        if not overlap:
            return False

        nested=(s1<=s2 and e2<=e1) or (s2<=s1 and e1 <=e2)

        return not nested

    def decode_one(self,scores,span_indices):
        candiates=[]

        N_span,E=scores.shape

        for n in range(N_span):
            for e in range(E):
                score=scores[n, e].item()

                if score>self.threshold:
                    start,end=span_indices[n]
                    label=self.entity_types[e]
                    candiates.append((start,end,label,score))

        candiates.sort(key=lambda x:x[3],reverse=True)

        selected=[]

        for cand in candiates:
            c_start,c_end,c_label,c_score=cand
            conflict=False

            for sel in selected:
                s_start,s_end,_,_=sel

                if self.mode=="flat":
                    if self.overlapping((c_start,c_end),(s_start,s_end)):
                        conflict=True

                        break

                elif self.mode=="nested":
                    if self.partial_overlap((c_start,c_end),(s_start,s_end)):
                        conflict=True

                        break

            if not conflict:
                selected.append(cand)

        return selected

    def decode_batch(self,scores,span_indices):
        results=[]
        B=scores.shape[0]

        for b in range(B):
            results.append(self.decode_one(scores[b],span_indices))

        return results

In [25]:
# test
decoder=Decoder(
    entity_types=["PER","ORG"],
    threshold=0.5,
    mode="flat"
)

scores=torch.tensor([
    [0.9,0.1],
    [0.8,0.2],
    [0.3,0.7],
    [0.6,0.4],
],dtype=torch.float32)

span_indices=[
    (0,0),
    (0,1),
    (2,2),
    (1,2),
]
print(scores.shape)
result=decoder.decode_one(scores,span_indices)
print(result)

torch.Size([4, 2])
[(0, 0, 'PER', 0.8999999761581421), (2, 2, 'ORG', 0.699999988079071)]


In [26]:
# test
assert len(result)==2
assert result[0][0:3]==(0,0,"PER")
assert result[1][0:3]==(2,2,"ORG")
print("pass")

pass


In [27]:
# test
scores_batch = torch.tensor([
    [
        [0.9, 0.1],
        [0.8, 0.2],
        [0.3, 0.7],
        [0.6, 0.4],
    ],
    [
        [0.2, 0.9],
        [0.1, 0.4],
        [0.7, 0.2],
        [0.3, 0.8],
    ]
], dtype=torch.float32)

decoder = Decoder(
    entity_types=["PER", "ORG"],
    threshold=0.5,
    mode="flat"
)

results = decoder.decode_batch(scores_batch, span_indices)
print(results)

[[(0, 0, 'PER', 0.8999999761581421), (2, 2, 'ORG', 0.699999988079071)], [(0, 0, 'ORG', 0.8999999761581421), (1, 2, 'ORG', 0.800000011920929)]]


In [28]:
decoder_nested = Decoder(
    entity_types=["PER", "ORG"],
    threshold=0.5,
    mode="nested"
)

scores = torch.tensor([
    [0.95, 0.1],   # (0,2) -> PER
    [0.85, 0.2],   # (1,1) -> PER   : (0,2) 안에 완전 포함
    [0.80, 0.1],   # (1,3) -> PER   : (0,2)와 partial overlap
], dtype=torch.float32)

span_indices = [
    (0, 2),
    (1, 1),
    (1, 3),
]

result_nested = decoder_nested.decode_one(scores, span_indices)
print(result_nested)

[(0, 2, 'PER', 0.949999988079071), (1, 1, 'PER', 0.8500000238418579)]


In [40]:
class GLiNER(nn.Module):
    def __init__(self,backbone_model,entity_types,d_ff,max_span_width=10,threshold=0.5,mode="flat"):
        super().__init__()

        self.entity_types=entity_types
        self.num_entity_types=len(entity_types)
        self.max_span_width=max_span_width

        self.encoder=EncoderModule(backbone_model)
        d_model=self.encoder.hidden_size

        self.text_token_rep=Text_Token_Represntation()
        self.ent_token_rep=ENT_Token_Representation()

        self.ent_rep=Entity_Representation(d_model,d_ff)
        self.span_rep=Span_Representation(d_model,d_ff)

        self.matching=Matching()

        self.decoder=Decoder(
            entity_types=entity_types,
            threshold=threshold,
            mode=mode
        )

    def build_span_representation(self,text_embeddings):
        B,T,H=text_embeddings.shape

        span_embeddings=[]
        span_indices=[]

        for start in range(T):
            max_end=min(T,start+self.max_span_width)
            for end in range(start,max_end):
                start_token=text_embeddings[:,start,:]
                end_token=text_embeddings[:,end,:]

                span_emb=self.span_rep(start_token,end_token)
                span_embeddings.append(span_emb)
                span_indices.append((start,end))

        span_embeddings=torch.stack(span_embeddings,dim=1)

        return span_embeddings,span_indices

    def forward(self,input_ids,attention_mask,text_mask,ent_mask):
        hidden_states=self.encoder(input_ids,attention_mask)

        text_embeddings=self.text_token_rep(hidden_states,text_mask)
        ent_embeddings=self.ent_token_rep(hidden_states,ent_mask)

        ent_vectors=self.ent_rep(ent_embeddings)
        span_vectors,span_indices=self.build_span_representation(text_embeddings)
        scores=self.matching(ent_vectors,span_vectors)
        return scores,span_indices

    @torch.no_grad()
    def predict(self,input_ids,attention_mask,text_mask,ent_mask):
        scores,span_indices=self.forward(input_ids=input_ids,attention_mask=attention_mask,text_mask=text_mask,ent_mask=ent_mask)
        probs=torch.sigmoid(scores)
        results=self.decoder.decode_batch(probs,span_indices)

        return results

In [30]:
model=GLiNER(
    backbone_model=BackBone_Model,
    entity_types=["person","location","organization"],
    d_ff=512,
    max_span_width=10,
    threshold=0.5,
    mode="flat"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
import json
with open("train_gliner-3.jsonl","r",encoding='utf-8') as f:
    sample=json.loads(f.readline())

In [32]:
input_ids = torch.tensor(sample["input_ids"]).unsqueeze(0)
attention_mask = torch.tensor(sample["attention_mask"]).unsqueeze(0)
text_mask = torch.tensor(sample["text_mask"]).unsqueeze(0)
ent_mask = torch.tensor(sample["ent_mask"]).unsqueeze(0)

In [33]:
import json
import torch

with open("train_gliner-3.jsonl", "r", encoding="utf-8") as f:
    sample = json.loads(f.readline())

input_ids = torch.tensor(sample["input_ids"]).unsqueeze(0)
attention_mask = torch.tensor(sample["attention_mask"]).unsqueeze(0)
text_mask = torch.tensor(sample["text_mask"]).unsqueeze(0)
ent_mask = torch.tensor(sample["ent_mask"]).unsqueeze(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

input_ids = input_ids.to(device)
attention_mask = attention_mask.to(device)
text_mask = text_mask.to(device)
ent_mask = ent_mask.to(device)

scores, span_indices = model(input_ids, attention_mask, text_mask, ent_mask)

print("scores shape:", scores.shape)
print("first 5 span indices:", span_indices[:5])

scores shape: torch.Size([1, 615, 6])
first 5 span indices: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4)]


In [34]:
hidden_states = model.encoder(input_ids, attention_mask)
print("hidden_states:", hidden_states.shape)

text_embeddings = model.text_token_rep(hidden_states, text_mask)
print("text_embeddings:", text_embeddings.shape)

ent_embeddings = model.ent_token_rep(hidden_states, ent_mask)
print("ent_embeddings:", ent_embeddings.shape)

entity_vectors = model.ent_rep(ent_embeddings)
print("entity_vectors:", entity_vectors.shape)

span_vectors, span_indices = model.build_span_representation(text_embeddings)
print("span_vectors:", span_vectors.shape)
print("len(span_indices):", len(span_indices))

hidden_states: torch.Size([1, 85, 768])
text_embeddings: torch.Size([1, 66, 768])
ent_embeddings: torch.Size([1, 6, 768])
entity_vectors: torch.Size([1, 6, 768])
span_vectors: torch.Size([1, 615, 768])
len(span_indices): 615


In [35]:
def build_target_matrix(span_indices, span_labels, num_entity_types, device=None):
    N_span = len(span_indices)
    E = num_entity_types

    target = torch.zeros(N_span, E, dtype=torch.float32, device=device)
    span_to_idx = {span: i for i, span in enumerate(span_indices)}

    for start, end, label_id in span_labels:
        span = (start, end)
        if span in span_to_idx:
            n = span_to_idx[span]
            target[n, label_id] = 1.0

    return target


def build_batch_target_matrix(span_indices, batch_span_labels, num_entity_types, device=None):
    targets = []
    for span_labels in batch_span_labels:
        target = build_target_matrix(
            span_indices=span_indices,
            span_labels=span_labels,
            num_entity_types=num_entity_types,
            device=device
        )
        targets.append(target)

    return torch.stack(targets, dim=0)

In [36]:
criterion = torch.nn.BCELoss()

scores, span_indices = model(input_ids, attention_mask, text_mask, ent_mask)

print("scores shape:", scores.shape)
print("len(span_indices):", len(span_indices))

targets = build_batch_target_matrix(
    span_indices=span_indices,
    batch_span_labels=[sample["span_labels"]],
    num_entity_types=scores.shape[-1],
    device=scores.device
)

print("targets shape:", targets.shape)

loss = criterion(scores, targets)
print("loss:", loss.item())

scores shape: torch.Size([1, 615, 6])
len(span_indices): 615
targets shape: torch.Size([1, 615, 6])
loss: 11.815717697143555


In [37]:
scores, span_indices = model(input_ids, attention_mask, text_mask, ent_mask)

print("scores shape:", scores.shape)
print("len(span_indices):", len(span_indices))

scores shape: torch.Size([1, 615, 6])
len(span_indices): 615


In [44]:
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


class GLiNERDataset(Dataset):
    def __init__(self, path):
        self.samples = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                self.samples.append(json.loads(line))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        return {
            "input_ids": torch.tensor(sample["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(sample["attention_mask"], dtype=torch.long),
            "text_mask": torch.tensor(sample["text_mask"], dtype=torch.long),
            "ent_mask": torch.tensor(sample["ent_mask"], dtype=torch.long),
            "span_labels": sample["span_labels"]
        }


def gliner_collate_fn(batch):
    x = batch[0]
    return {
        "input_ids": x["input_ids"].unsqueeze(0),
        "attention_mask": x["attention_mask"].unsqueeze(0),
        "text_mask": x["text_mask"].unsqueeze(0),
        "ent_mask": x["ent_mask"].unsqueeze(0),
        "span_labels": [x["span_labels"]]
    }


def build_target_matrix(span_indices, span_labels, num_entity_types, device=None):
    n_span = len(span_indices)
    target = torch.zeros(n_span, num_entity_types, dtype=torch.float32, device=device)

    span_to_idx = {span: i for i, span in enumerate(span_indices)}

    for start, end, label_id in span_labels:
        if not (0 <= label_id < num_entity_types):
            raise ValueError(f"label_id {label_id} out of range for num_entity_types={num_entity_types}")

        span = (start, end)
        if span in span_to_idx:
            target[span_to_idx[span], label_id] = 1.0

    return target


def build_batch_target_matrix(span_indices, batch_span_labels, num_entity_types, device=None):
    targets = []
    for span_labels in batch_span_labels:
        target = build_target_matrix(
            span_indices=span_indices,
            span_labels=span_labels,
            num_entity_types=num_entity_types,
            device=device
        )
        targets.append(target)
    return torch.stack(targets, dim=0)


def debug_one_batch(model, dataloader, criterion, device):
    model.train()

    batch = next(iter(dataloader))

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    text_mask = batch["text_mask"].to(device)
    ent_mask = batch["ent_mask"].to(device)
    batch_span_labels = batch["span_labels"]

    scores, span_indices = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        text_mask=text_mask,
        ent_mask=ent_mask
    )

    targets = build_batch_target_matrix(
        span_indices=span_indices,
        batch_span_labels=batch_span_labels,
        num_entity_types=scores.shape[-1],
        device=device
    )

    print("scores shape:", scores.shape)
    print("len(span_indices):", len(span_indices))
    print("targets shape:", targets.shape)
    print("target sum:", targets.sum().item())
    print("score min/max:", scores.min().item(), scores.max().item())

    if torch.isnan(scores).any():
        raise ValueError("scores has NaN")
    if torch.isnan(targets).any():
        raise ValueError("targets has NaN")

    loss = criterion(scores, targets)
    print("loss:", loss.item())


def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for step, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        text_mask = batch["text_mask"].to(device)
        ent_mask = batch["ent_mask"].to(device)
        batch_span_labels = batch["span_labels"]

        scores, span_indices = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            text_mask=text_mask,
            ent_mask=ent_mask
        )

        targets = build_batch_target_matrix(
            span_indices=span_indices,
            batch_span_labels=batch_span_labels,
            num_entity_types=scores.shape[-1],
            device=device
        )

        if torch.isnan(scores).any():
            raise ValueError(f"scores has NaN at step {step}")
        if torch.isnan(targets).any():
            raise ValueError(f"targets has NaN at step {step}")

        loss = criterion(scores, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if step % 50 == 0:
            print(f"step {step} | loss {loss.item():.4f}")

    return total_loss / len(dataloader)


train_path = "/content/train_gliner-3.jsonl"
train_dataset = GLiNERDataset(train_path)
train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=gliner_collate_fn
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

debug_one_batch(model, train_loader, criterion, device)

num_epochs = 3
for epoch in range(num_epochs):
    avg_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    print(f"Epoch {epoch+1}/{num_epochs} - avg loss: {avg_loss:.4f}")

scores shape: torch.Size([1, 475, 6])
len(span_indices): 475
targets shape: torch.Size([1, 475, 6])
target sum: 4.0
score min/max: nan nan


ValueError: scores has NaN